In [1]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [26]:
import torch 
import torch.nn as nn
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

weights = torch.randn(2, 2) 
biases = torch.zeros(2)


data = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
data

data = data.to(device)

np_arr = np.array([[1.0, 2.0], [3.0, 4.0]])
t = torch.from_numpy(np_arr)
t[1, 0] = 11

print(np_arr[1])

dot_prod = torch.matmul(data, weights) + biases
dot_prod

Using cpu device
[11.  4.]


tensor([[ -4.2464,   1.2996],
        [-10.0871,   1.0256]])

In [55]:
# ...existing code...
data = torch.arange(12).reshape(3,4)
data

rows = torch.tensor([0, 2])
cols = torch.tensor([1, 3]) 



mask = (data % 2 == 0)
even_numbers = data[mask]

print("Even numbers:", even_numbers)

batched_data = data.unsqueeze(0)
batched_data

Even numbers: tensor([ 0,  2,  4,  6,  8, 10])


tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]]])

In [57]:
x = torch.tensor([2.0, 3.0], requires_grad=True)

# Forward pass: Define an arbitrary function
# Example: y = sum(x^2 + 5x)
y = (x**2 + 5*x).sum()

y.backward()

print("Gradients:", x.grad) 

# IMPORTANT: PyTorch accumulates gradients by default. 
# If you run a loop, you must zero them out before the next pa

x.grad.zero_()

# Context Managers: Stopping the tracking
# Use this during inference/testing to save memory and compute
with torch.no_grad():
    predictions = (x**2).sum() # No graph is built here

Gradients: tensor([ 9., 11.])


In [60]:
import torch.nn as torch_nn
import torch.nn.functional as F

class CustomClassifier(torch_nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(CustomClassifier, self).__init__() # Initialize the parent class
        
        # Define the layers (these contain learnable weights)
        self.layer1 = torch_nn.Linear(input_dim, hidden_dim)
        self.layer2 = torch_nn.Linear(hidden_dim, hidden_dim)
        self.output = torch_nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        # Define how data flows through the architecture
        # F.relu is the stateless activation function
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        
        # No softmax here; PyTorch's CrossEntropyLoss applies it internally
        logits = self.output(x) 
        return logits

# Instantiate the model
model = CustomClassifier(input_dim=10, hidden_dim=32, num_classes=3)
model

CustomClassifier(
  (layer1): Linear(in_features=10, out_features=32, bias=True)
  (layer2): Linear(in_features=32, out_features=32, bias=True)
  (output): Linear(in_features=32, out_features=3, bias=True)
)

In [71]:
import torch.nn as nn
import torch.nn.functional as F

class CustomerClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(CustomerClassifier, self).__init__()

        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        x = F.relu(self.layer3(x))

        logits = self.output(x)
        return logits

model = CustomerClassifier(input_dim=10, hidden_dim=32, num_classes=3)
model

CustomerClassifier(
  (layer1): Linear(in_features=10, out_features=32, bias=True)
  (layer2): Linear(in_features=32, out_features=32, bias=True)
  (layer3): Linear(in_features=32, out_features=32, bias=True)
  (output): Linear(in_features=32, out_features=3, bias=True)
)

In [72]:
import torch.optim as optim

# 1. Setup Data, Optimizer, and Loss Function
dummy_inputs = torch.randn(16, 10)  # Batch of 16, 10 features each
dummy_labels = torch.randint(0, 3, (16,)) # 16 labels (classes 0, 1, 2)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = torch_nn.CrossEntropyLoss()

epochs = 5

for epoch in range(epochs):
    # Step 1: Zero out the gradients from the previous batch
    optimizer.zero_grad()
    
    # Step 2: Forward pass (computes predictions)
    predictions = model(dummy_inputs)
    
    # Step 3: Compute the loss against true labels
    loss = criterion(predictions, dummy_labels)
    
    # Step 4: Backward pass (computes gradients via autograd)
    loss.backward()
    
    # Step 5: Update the network weights
    optimizer.step()
    
    print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")

Epoch 1 | Loss: 1.0591
Epoch 2 | Loss: 1.0547
Epoch 3 | Loss: 1.0503
Epoch 4 | Loss: 1.0460
Epoch 5 | Loss: 1.0418
